In [ ]:
pip install open_clip_torch -q

In [ ]:
# pip install -qU "transformers>=4.56.0" "huggingface_hub>=0.24.0" # comment if don't use dinov3

In [ ]:
import os
from pathlib import Path
from dataclasses import dataclass
from typing import List, Optional

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import umap
from matplotlib.patches import Patch
from sklearn.manifold import TSNE

import torch.nn.functional as F
from torchvision import models
from torchvision.models import ResNet18_Weights, ResNet50_Weights

from transformers import AutoTokenizer, AutoModel, AutoImageProcessor, AutoProcessor
import open_clip

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

In [ ]:
@dataclass
class DatasetInfo:
    csv_path: str
    data_column: str
    label_column: str
    data_type: str  # 'text' or 'image'
    output_dir: str
    dataset_name: str
    image_dir: Optional[str] = None
    label_description_path: Optional[str] = None

In [ ]:
class EmbeddingGenerator:
    """
    Generate embeddings for text or image data and save the results.
    """
    MODEL_MAP = {
        # text
        "bert":    "bert-base-uncased",
        "bge":     "BAAI/bge-m3",
        "xlnet":   "xlnet/xlnet-base-cased",
        "roberta": "FacebookAI/roberta-large",

        # image
        "clip-base-16": "ViT-B-16",               # open_clip
        "SigLIP-webli": "ViT-SO400M-14-SigLIP",   # open_clip
        "siglip2":      "google/siglip2-so400m-patch16-naflex",
        "dinov2":       "facebook/dinov2-large",
        "dinov3":       "facebook/dinov3-vitl16-pretrain-lvd1689m",
        "resnet18":     "resnet18",               # torchvision
        "resnet50":     "resnet50",               # torchvision
    }

    CLIP_WEIGHTS = {
        "clip-base-16": "datacomp_xl_s13b_b90k",
        "SigLIP-webli": "webli"
    }

    def __init__(self, dataset_info, model_name: str, batch_size: int):
        self.dataset_info = dataset_info
        self.model_name = model_name
        self.batch_size = batch_size
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self._validate_input()

        print(f"Loading model '{self.model_name}' on device '{self.device}'...")

        if self.dataset_info.data_type == 'text':
            model_id = self.MODEL_MAP[self.model_name]
            self.tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
            self.model = AutoModel.from_pretrained(model_id).to(self.device)

        elif self.dataset_info.data_type == 'image':
            if "clip" in self.model_name or self.model_name == "SigLIP-webli":
                model_id = self.MODEL_MAP[self.model_name]
                pretrained = self.CLIP_WEIGHTS[self.model_name]
                self.model, _, self.preprocess = open_clip.create_model_and_transforms(
                    model_name=model_id,
                    pretrained=pretrained,
                    device=self.device
                )
            elif self.model_name == "dinov2":
                model_id = self.MODEL_MAP[self.model_name]
                self.processor = AutoImageProcessor.from_pretrained(model_id)
                self.model = AutoModel.from_pretrained(model_id).to(self.device)
            elif self.model_name == "dinov3":
                model_id = self.MODEL_MAP[self.model_name]
                self.processor = AutoImageProcessor.from_pretrained(model_id, token=HF_TOKEN)
                self.model = AutoModel.from_pretrained(model_id,token=HF_TOKEN).to(self.device)
            elif self.model_name == "siglip2":
                model_id = self.MODEL_MAP[self.model_name]
                self.processor = AutoProcessor.from_pretrained(model_id)
                self.model = AutoModel.from_pretrained(model_id).to(self.device)
            elif self.model_name in ("resnet18", "resnet50"):
                if self.model_name == "resnet18":
                    weights = ResNet18_Weights.DEFAULT
                    model = models.resnet18(weights=weights)
                else:
                    weights = ResNet50_Weights.DEFAULT
                    model = models.resnet50(weights=weights)
                # lấy feature trước lớp FC
                model.fc = torch.nn.Identity()
                self.preprocess = weights.transforms()
                self.model = model.to(self.device)
            else:
                raise ValueError(f"Unsupported image model: {self.model_name}")

        self.model.eval()
        print("Model loaded successfully.")

    def _validate_input(self):
        if self.model_name not in self.MODEL_MAP:
            raise ValueError(f"Invalid model name '{self.model_name}'. "
                             f"Available options: {list(self.MODEL_MAP.keys())}")

        if self.dataset_info.data_type == 'text':
            if self.model_name not in {'bert', 'bge', 'xlnet', 'roberta'}:
                raise ValueError("For text data, model_name must be one of: 'bert', 'bge', 'xlnet', 'roberta'.")

        if self.dataset_info.data_type == 'image':
            if self.model_name in {'bert', 'bge', 'xlnet', 'roberta'}:
                raise ValueError(f"Model '{self.model_name}' is not suitable for image data.")
            if not self.dataset_info.image_dir:
                raise ValueError("'image_dir' must be provided for image data.")

    def _mean_pooling(self, model_output, attention_mask):
        token_embeddings = model_output.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

    # -------- TEXT --------
    def _embed_text(self, texts: List[str]) -> np.ndarray:
        inputs = self.tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(self.device)
        with torch.no_grad():
            outputs = self.model(**inputs)
            
        if self.model_name == "bge":
            # BGE-M3: dùng CLS + L2 normalize cho dense embedding (khuyến nghị chính thức)
            cls = outputs.last_hidden_state[:, 0, :]
            embeddings = F.normalize(cls, p=2, dim=1)
        else:
            # BERT / RoBERTa / XLNet: mặc định dùng mean pooling
            embeddings = self._mean_pooling(outputs, inputs['attention_mask'])
            # --- (tuỳ chọn) Dùng CLS thay vì mean pooling ---
            # cls = outputs.last_hidden_state[:, 0, :]
            # embeddings = F.normalize(cls, p=2, dim=1)

        return embeddings.cpu().numpy()

    # -------- IMAGE --------
    def _embed_images(self, image_names: List[str]) -> np.ndarray:
        image_paths = [Path(self.dataset_info.image_dir) / name for name in image_names]
        try:
            images = [Image.open(p).convert("RGB") for p in image_paths]
        except FileNotFoundError as e:
            print(f"Error: image file not found - {e}")
            raise

        with torch.no_grad():
            if "clip" in self.model_name or self.model_name == "SigLIP-webli":
                image_tensors = torch.stack([self.preprocess(img) for img in images]).to(self.device)
                outputs = self.model.encode_image(image_tensors)

            elif self.model_name == "dinov2":
                inputs = self.processor(images=images, return_tensors="pt").to(self.device)
                hidden = self.model(**inputs).last_hidden_state
                # mean pooling qua patch tokens
                # outputs = hidden.mean(dim=1)
                # --- Dùng CLS theo gợi ý HF cho classification/retrieval ---
                outputs = hidden[:, 0, :]

            elif self.model_name == "dinov3":
                inputs = self.processor(images=images, return_tensors="pt").to(self.device)
                with torch.no_grad():
                    outputs = self.model(**inputs)
                # Mặc định dùng pooled embedding (khuyến nghị trong docs)
                outputs = outputs.pooler_output
                # --- (tùy chọn) Dùng CLS token thay cho pooled ---
                # hidden = outputs.last_hidden_state
                # outputs = hidden[:, 0, :]
                # --- (tùy chọn) Mean pooling trên patch tokens (bỏ CLS + register tokens) ---
                # patch_tokens = hidden[:, 1 + self.model.config.num_register_tokens:, :]
                # outputs = patch_tokens.mean(dim=1)
            
            elif self.model_name == "siglip2":
                inputs = self.processor(images=images, return_tensors="pt").to(self.device)
                outputs = self.model.get_image_features(**inputs)  # (batch, dim)

            elif self.model_name in ("resnet18", "resnet50"):
                image_tensors = torch.stack([self.preprocess(img) for img in images]).to(self.device)
                outputs = self.model(image_tensors)  # (batch, 512) hoặc (batch, 2048)

            else:
                raise ValueError("Unsupported image model.")

        return outputs.cpu().numpy()

    def generate(self) -> str:
        output_dir = Path(self.dataset_info.output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        file_name = f"{self.dataset_info.dataset_name}_embedding_{self.model_name}.feather"
        output_path = output_dir / file_name

        print(f"Starting embedding process. Output will be saved to: {output_path}")

        df = pd.read_csv(self.dataset_info.csv_path)
        data_to_process = df[self.dataset_info.data_column].tolist()

        all_embeddings = []

        for i in tqdm(range(0, len(data_to_process), self.batch_size), desc=f"Embedding with {self.model_name}"):
            batch_data = data_to_process[i:i + self.batch_size]

            if self.dataset_info.data_type == 'text':
                batch_embeddings = self._embed_text(batch_data)
            else:
                batch_embeddings = self._embed_images(batch_data)

            all_embeddings.append(batch_embeddings)

        final_embeddings = np.vstack(all_embeddings)
        embedding_df = pd.DataFrame(final_embeddings, columns=[str(i) for i in range(final_embeddings.shape[1])])
        print(f"Shape of embedding: {embedding_df.shape}")
        embedding_df.index.name = 'index'

        embedding_df.reset_index(inplace=True)
        embedding_df.to_feather(output_path)

        print("Done! Embedding file has been saved.")
        return str(output_path)


In [ ]:
def plot_tsne_or_umap(
    dataset_name: str,
    csv_path: str,
    label_col: str,
    feather_path: str,
    class_names: list,
    mode: str = "tsne",  # "tsne" or "umap"
    max_characters: int = 20
):
    """
    Plot a t-SNE or UMAP scatter plot with class index and name in the legend.

    Args:
    - dataset_name: name of the dataset
    - csv_path: path to CSV file containing true labels
    - label_col: name of the true label column in the CSV
    - feather_path: path to feather file containing embeddings and index
    - class_names: list of class names ordered by label index (0, 1, 2, ...)
    - mode: "tsne" or "umap"
    """
    assert mode in {"tsne", "umap"}, "mode must be 'tsne' or 'umap'"

    # Load data
    df_csv = pd.read_csv(csv_path).reset_index().rename(columns={label_col: "true_label"})
    df_feather = pd.read_feather(feather_path)
    df_merged = df_feather.merge(df_csv[['index', 'true_label']], on='index', how='left')

    # Extract embeddings and labels
    embed_cols = [col for col in df_feather.columns if col != "index"]
    X = df_merged[embed_cols].values
    y_true = df_merged["true_label"].values

    # Define color palette based on class labels
    class_ids = sorted(np.unique(y_true))
    palette = sns.color_palette("hls", len(class_ids))
    color_map = {cls: palette[i] for i, cls in enumerate(class_ids)}

    # Dimensionality reduction
    if mode == "tsne":
        print("🔍 Running t-SNE...")
        X_proj = TSNE(n_components=2, random_state=42, init='pca').fit_transform(X)
        title = f"{dataset_name} - true_label - t-SNE"
    else:
        print("🔍 Running UMAP...")
        X_proj = umap.UMAP(n_components=2, random_state=42).fit_transform(X)
        title = f"{dataset_name} - true_label - UMAP"

    # Plot
    plt.figure(figsize=(20, 14))
    ax = sns.scatterplot(
        x=X_proj[:, 0],
        y=X_proj[:, 1],
        hue=y_true,
        palette=color_map,
        s=10,
        alpha=0.8,
        linewidth=0,
        legend=False
    )
    plt.title(title, fontsize=16)
    plt.xlabel("dim 1")
    plt.ylabel("dim 2")

    # Construct legend with label index and truncated class name
    legend_elements = [
        Patch(facecolor=color_map[cls], label=f"{cls} - {class_names[cls][:max_characters]}")
        for cls in class_ids
    ]
    plt.legend(
        handles=legend_elements,
        title="Class",
        loc="center left",
        bbox_to_anchor=(1, 0.5),
        borderaxespad=0.
    )
    plt.tight_layout()
    plt.show()


In [ ]:
# ------------------- CONFIG & RUN -------------------k
config = {
    "dataset_name": "finance-test-0.5k",
    "data_type": "text",  # image or text
    "csv_path": "/kaggle/input/finance-snorkel-mlp-raw/finance_test_Snorkel_MLP_0.csv",
    "image_dir": None, # None if data_type is text
    "output_dir": "/kaggle/working/",
    "data_column": "text",
    "label_column": "label",
    "model_name": "bert",  
    # Options in model: 
    # text : [bert, bge, xlnet, roberta]  
    # image : [clip-base-16, dinov2, dinov3, siglip2, SigLIP-webli, resnet18, resnet50]
    "batch_size": 512,
    "plot_mode": "umap",  # or "tsne"
    "class_names": [
        "Class_01", "Class_02", "Class_03",
    ]
}

# Khởi tạo dataset info
dataset_info = DatasetInfo(
    csv_path=config["csv_path"],
    data_column=config["data_column"],
    label_column=config["label_column"],
    data_type=config["data_type"],
    output_dir=config["output_dir"],
    dataset_name=config["dataset_name"],
    image_dir=config["image_dir"]
)

# Sinh embedding
embed_gen = EmbeddingGenerator(
    dataset_info=dataset_info,
    model_name=config["model_name"],
    batch_size=config["batch_size"]
)

embedding_path = embed_gen.generate()
print("Embedding file saved to:", embedding_path)

# Vẽ UMAP hoặc t-SNE
plot_tsne_or_umap(
    dataset_name=config["dataset_name"],
    class_names=config["class_names"],
    csv_path=config["csv_path"],
    label_col=config["label_column"],
    feather_path=embedding_path,
    mode=config["plot_mode"]
)

In [ ]:
df = pd.read_feather(embedding_path)
df

In [ ]:
df.columns